In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(6):
    if os.path.isdir(os.path.join(_root, "tools")) and os.path.isdir(os.path.join(_root, "data")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from tools.paths import data_path, outputs_path

In [2]:
# --- QCEW scoping test: verify structure before the full pull ---
#
# I can't reach data.bls.gov from my own environment to test this myself,
# so treat this cell as the real verification step - run it and check
# what actually comes back before we build the full 5-county x 5-year
# historical pull.

import pandas as pd
import requests

NYC_COUNTY_FIPS = {
    "Manhattan (New York County)": "36061",
    "Bronx": "36005",
    "Brooklyn (Kings County)": "36047",
    "Queens": "36081",
    "Staten Island (Richmond County)": "36085",
}

# Quarterly field layout, from the BLS QCEW CSV Data Slices documentation.
QCEW_QUARTERLY_COLUMNS = [
    "area_fips", "own_code", "industry_code", "agglvl_code", "size_code",
    "year", "qtr", "disclosure_code", "qtrly_estabs",
    "month1_emplvl", "month2_emplvl", "month3_emplvl",
    "total_qtrly_wages", "taxable_qtrly_wages", "qtrly_contributions",
    "avg_wkly_wage", "lq_disclosure_code", "lq_qtrly_estabs",
    "lq_month1_emplvl", "lq_month2_emplvl", "lq_month3_emplvl",
    "lq_total_qtrly_wages", "lq_taxable_qtrly_wages", "lq_qtrly_contributions",
    "lq_avg_wkly_wage", "oty_disclosure_code",
    "oty_qtrly_estabs_chg", "oty_qtrly_estabs_pct_chg",
    "oty_month1_emplvl_chg", "oty_month1_emplvl_pct_chg",
    "oty_month2_emplvl_chg", "oty_month2_emplvl_pct_chg",
    "oty_month3_emplvl_chg", "oty_month3_emplvl_pct_chg",
    "oty_total_qtrly_wages_chg", "oty_total_qtrly_wages_pct_chg",
    "oty_taxable_qtrly_wages_chg", "oty_taxable_qtrly_wages_pct_chg",
    "oty_qtrly_contributions_chg", "oty_qtrly_contributions_pct_chg",
    "oty_avg_wkly_wage_chg", "oty_avg_wkly_wage_pct_chg",
]

# Annual-average field layout is shorter (no monthly breakdown).
QCEW_ANNUAL_COLUMNS = [
    "area_fips", "own_code", "industry_code", "agglvl_code", "size_code",
    "year", "qtr", "disclosure_code", "annual_avg_estabs", "annual_avg_emplvl",
    "total_annual_wages", "taxable_annual_wages", "annual_contributions",
    "annual_avg_wkly_wage", "avg_annual_pay", "lq_disclosure_code",
    "lq_annual_avg_estabs", "lq_annual_avg_emplvl", "lq_total_annual_wages",
    "lq_taxable_annual_wages", "lq_annual_contributions", "lq_annual_avg_wkly_wage",
    "lq_avg_annual_pay", "oty_disclosure_code",
    "oty_annual_avg_estabs_chg", "oty_annual_avg_estabs_pct_chg",
    "oty_annual_avg_emplvl_chg", "oty_annual_avg_emplvl_pct_chg",
    "oty_total_annual_wages_chg", "oty_total_annual_wages_pct_chg",
    "oty_taxable_annual_wages_chg", "oty_taxable_annual_wages_pct_chg",
    "oty_annual_contributions_chg", "oty_annual_contributions_pct_chg",
    "oty_annual_avg_wkly_wage_chg", "oty_annual_avg_wkly_wage_pct_chg",
    "oty_avg_annual_pay_chg", "oty_avg_annual_pay_pct_chg",
]


def fetch_qcew_area_slice(year: int, qtr: str, area_fips: str) -> pd.DataFrame:
    """
    qtr: '1'-'4' for a single quarter, or 'a' for annual average.
    Returns a DataFrame with the file's header row used as-is (BLS CSVs
    do include a header row, unlike the quirky exports we've dealt with
    from Airtable/InPlace so far).
    """
    url = f"https://data.bls.gov/cew/data/api/{year}/{qtr}/area/{area_fips}.csv"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    from io import StringIO
    return pd.read_csv(StringIO(resp.text))


# --- Scoping test: one county, one year, annual average ---
TEST_YEAR = 2025  # should be safely published by now
TEST_COUNTY_NAME = "Manhattan (New York County)"
TEST_COUNTY_FIPS = NYC_COUNTY_FIPS[TEST_COUNTY_NAME]

print(f"Pulling {TEST_COUNTY_NAME} ({TEST_COUNTY_FIPS}), {TEST_YEAR} annual average...")
df = fetch_qcew_area_slice(TEST_YEAR, "a", TEST_COUNTY_FIPS)

print(f"\nRows returned: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

print("\nUnique aggregation levels (agglvl_code) present:")
print(df["agglvl_code"].value_counts().sort_index().to_string())

print("\nUnique ownership codes (own_code) present:")
print(df["own_code"].value_counts().sort_index().to_string())

# agglvl_code 74 = County, NAICS Sector, by Ownership (confirmed from the
# actual BLS aggregation-level matrix - county-level codes run 70-78, not
# the national-level 10-19 codes I'd guessed initially). own_code 5 =
# Private ownership - the right slice since internship placements are
# overwhelmingly at private-sector and nonprofit employers (QCEW counts
# nonprofits under Private, not Government).
sector_level = df[(df["agglvl_code"] == 74) & (df["own_code"] == 5)]
print(f"\nSector-level (agglvl_code=74), Private ownership (own_code=5) rows: {len(sector_level)}")
print(sector_level[["industry_code", "annual_avg_estabs", "annual_avg_emplvl", "disclosure_code"]].to_string(index=False))

n_suppressed = (sector_level["disclosure_code"] == "N").sum()
print(f"\nSuppressed (disclosure_code='N') sector-level rows: {n_suppressed} of {len(sector_level)}")
print("If this is 0 or small, sector-level is a safe granularity to use for the full pull.")

Pulling Manhattan (New York County) (36061), 2025 annual average...

Rows returned: 2014
Columns: ['area_fips', 'own_code', 'industry_code', 'agglvl_code', 'size_code', 'year', 'qtr', 'disclosure_code', 'annual_avg_estabs', 'annual_avg_emplvl', 'total_annual_wages', 'taxable_annual_wages', 'annual_contributions', 'annual_avg_wkly_wage', 'avg_annual_pay', 'lq_disclosure_code', 'lq_annual_avg_estabs', 'lq_annual_avg_emplvl', 'lq_total_annual_wages', 'lq_taxable_annual_wages', 'lq_annual_contributions', 'lq_annual_avg_wkly_wage', 'lq_avg_annual_pay', 'oty_disclosure_code', 'oty_annual_avg_estabs_chg', 'oty_annual_avg_estabs_pct_chg', 'oty_annual_avg_emplvl_chg', 'oty_annual_avg_emplvl_pct_chg', 'oty_total_annual_wages_chg', 'oty_total_annual_wages_pct_chg', 'oty_taxable_annual_wages_chg', 'oty_taxable_annual_wages_pct_chg', 'oty_annual_contributions_chg', 'oty_annual_contributions_pct_chg', 'oty_annual_avg_wkly_wage_chg', 'oty_annual_avg_wkly_wage_pct_chg', 'oty_avg_annual_pay_chg', 'oty_

In [5]:
# --- Full QCEW pull: NYC citywide, sector-level, private employment,
#     2022-2025 annual averages ---
#
# Confirmed via the scoping test: agglvl_code=74 (County, NAICS Sector,
# by Ownership) + own_code=5 (Private) gives clean, unsuppressed data for
# all 20 NAICS sectors in Manhattan. Assuming the same holds for the other
# 4 boroughs (checked below, not assumed).
#
# 2026 is deliberately excluded - QCEW annual data needs all 4 quarters,
# and Q4 2026 won't be published until ~5 months after it ends (2027).

import pandas as pd
import requests
import time
from io import StringIO

NYC_COUNTY_FIPS = {
    "Manhattan": "36061",
    "Bronx": "36005",
    "Brooklyn": "36047",
    "Queens": "36081",
    "Staten Island": "36085",
}

YEARS = [2022, 2023, 2024, 2025]  # 2026 excluded - not fully published yet

TARGET_AGGLVL_CODE = 74  # County, NAICS Sector, by Ownership
TARGET_OWN_CODE = 5      # Private

SECONDS_BETWEEN_REQUESTS = 0.5

# Standard 2-digit NAICS sector titles, using QCEW's own sector codes
# (some sectors are combined, e.g. "31-33" for all of Manufacturing).
QCEW_SECTOR_TITLES = {
    "11": "Agriculture, Forestry, Fishing and Hunting",
    "21": "Mining, Quarrying, and Oil and Gas Extraction",
    "22": "Utilities",
    "23": "Construction",
    "31-33": "Manufacturing",
    "42": "Wholesale Trade",
    "44-45": "Retail Trade",
    "48-49": "Transportation and Warehousing",
    "51": "Information",
    "52": "Finance and Insurance",
    "53": "Real Estate and Rental and Leasing",
    "54": "Professional and Technical Services",
    "55": "Management of Companies and Enterprises",
    "56": "Administrative and Waste Services",
    "61": "Educational Services",
    "62": "Health Care and Social Assistance",
    "71": "Arts, Entertainment, and Recreation",
    "72": "Accommodation and Food Services",
    "81": "Other Services (except Public Administration)",
    "92": "Public Administration",
    "99": "Unclassified",
}


def naics_to_qcew_sector(naics_code) -> str:
    """
    Rolls up a (possibly 2-6 digit) NAICS code to the QCEW sector code it
    belongs to, handling the merged sectors (31-33, 44-45, 48-49).
    Use this on your internship opportunities' NAICS Code column to bring
    them to the same granularity as this QCEW pull, for comparison.
    """
    if pd.isna(naics_code):
        return None
    code = str(naics_code).strip()
    prefix2 = code[:2]
    if prefix2 in ("31", "32", "33"):
        return "31-33"
    if prefix2 in ("44", "45"):
        return "44-45"
    if prefix2 in ("48", "49"):
        return "48-49"
    return prefix2


def fetch_qcew_area_slice(year: int, qtr: str, area_fips: str) -> pd.DataFrame:
    url = f"https://data.bls.gov/cew/data/api/{year}/{qtr}/area/{area_fips}.csv"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return pd.read_csv(StringIO(resp.text))


def pull_nyc_sector_data() -> pd.DataFrame:
    all_rows = []
    for year in YEARS:
        for borough, fips in NYC_COUNTY_FIPS.items():
            print(f"Pulling {borough} ({fips}), {year}...")
            df = fetch_qcew_area_slice(year, "a", fips)
            sector = df[(df["agglvl_code"] == TARGET_AGGLVL_CODE) & (df["own_code"] == TARGET_OWN_CODE)].copy()

            n_suppressed = (sector["disclosure_code"] == "N").sum()
            if n_suppressed:
                print(f"  NOTE: {n_suppressed} of {len(sector)} sectors suppressed for {borough} {year}")

            sector["Borough"] = borough
            sector["Cohort Year"] = year
            all_rows.append(sector[["Borough", "Cohort Year", "industry_code",
                                     "annual_avg_estabs", "annual_avg_emplvl",
                                     "avg_annual_pay", "disclosure_code"]])
            time.sleep(SECONDS_BETWEEN_REQUESTS)

    combined = pd.concat(all_rows, ignore_index=True)
    combined = combined.rename(columns={
        "industry_code": "QCEW Sector Code",
        "annual_avg_estabs": "Establishment Count",
        "annual_avg_emplvl": "Average Annual Employment",
        "avg_annual_pay": "Average Annual Pay",
        "disclosure_code": "Disclosure Code",
    })
    combined["QCEW Sector Title"] = combined["QCEW Sector Code"].map(QCEW_SECTOR_TITLES)
    return combined


def aggregate_to_citywide(borough_level: pd.DataFrame) -> pd.DataFrame:
    """
    Sums employment and establishment counts across the 5 boroughs to get
    a single NYC citywide number per sector per year. Average Annual Pay
    can't just be summed or averaged across boroughs without weighting by
    employment, so it's dropped at this level - use the borough-level file
    if per-borough pay comparisons matter.
    """
    citywide = (
        borough_level.groupby(["Cohort Year", "QCEW Sector Code", "QCEW Sector Title"])
        .agg(
            Total_NYC_Establishment_Count=("Establishment Count", "sum"),
            Total_NYC_Average_Annual_Employment=("Average Annual Employment", "sum"),
            Boroughs_Suppressed=("Disclosure Code", lambda s: (s == "N").sum()),
        )
        .reset_index()
    )
    total_per_year = citywide.groupby("Cohort Year")["Total_NYC_Average_Annual_Employment"].transform("sum")
    citywide["Pct_Of_NYC_Total_Employment"] = (
        citywide["Total_NYC_Average_Annual_Employment"] / total_per_year * 100
    ).round(2)
    citywide["Contains Suppressed Borough Data"] = citywide["Boroughs_Suppressed"] > 0
    return citywide.sort_values(["Cohort Year", "Total_NYC_Average_Annual_Employment"], ascending=[True, False])


# --- Run the full pull ---
borough_level = pull_nyc_sector_data()
borough_level.to_csv(data_path("qcew_nyc_sector_by_borough.csv"), index=False)
print(f"\nWrote data/qcew_nyc_sector_by_borough.csv ({len(borough_level)} rows)")

citywide = aggregate_to_citywide(borough_level)
citywide.to_csv(data_path("qcew_nyc_sector_citywide.csv"), index=False)
print(f"Wrote data/qcew_nyc_sector_citywide.csv ({len(citywide)} rows)")

n_any_suppressed = (citywide["Boroughs_Suppressed"] > 0).sum()
print(f"\nSector-years with at least one borough suppressed: {n_any_suppressed} of {len(citywide)}")

print("\nCitywide employment by sector, most recent year available:")
latest_year = citywide["Cohort Year"].max()
print(citywide[citywide["Cohort Year"] == latest_year]
      [["QCEW Sector Title", "Total_NYC_Average_Annual_Employment", "Pct_Of_NYC_Total_Employment"]]
      .to_string(index=False))

Pulling Manhattan (36061), 2022...
Pulling Bronx (36005), 2022...
  NOTE: 4 of 20 sectors suppressed for Bronx 2022
Pulling Brooklyn (36047), 2022...
Pulling Queens (36081), 2022...
Pulling Staten Island (36085), 2022...
Pulling Manhattan (36061), 2023...
Pulling Bronx (36005), 2023...
  NOTE: 4 of 20 sectors suppressed for Bronx 2023
Pulling Brooklyn (36047), 2023...
Pulling Queens (36081), 2023...
Pulling Staten Island (36085), 2023...
  NOTE: 2 of 19 sectors suppressed for Staten Island 2023
Pulling Manhattan (36061), 2024...
Pulling Bronx (36005), 2024...
  NOTE: 4 of 20 sectors suppressed for Bronx 2024
Pulling Brooklyn (36047), 2024...
Pulling Queens (36081), 2024...
Pulling Staten Island (36085), 2024...
  NOTE: 2 of 19 sectors suppressed for Staten Island 2024
Pulling Manhattan (36061), 2025...
Pulling Bronx (36005), 2025...
  NOTE: 4 of 20 sectors suppressed for Bronx 2025
Pulling Brooklyn (36047), 2025...
Pulling Queens (36081), 2025...
Pulling Staten Island (36085), 2025...


## This notebook is just the public QCEW pull now

Trimmed 2026-08-15: this used to also carry the internal-vs-citywide comparison (4 cells -- a first-draft comparison, a growth-quadrant exploration, a multi-year trend view, and a raw-numbers view). All four predated weighting placements by `Max Placements` and were erroneous for it -- they counted 1 opportunity = 1 placement, undercounting every high-capacity posting. `placement_join/placements_analysis.ipynb`'s Ground Truth Analysis replaced all four with a single, Max-Placements-weighted version. That's the one to use -- this notebook is now just the QCEW fetch (cells above) that feeds `qcew_nyc_sector_citywide.csv` into it.